https://www.kaggle.com/datasets/saroz014/plant-disease/data

Для экономии ресурсов возьмем только яблони для построения модели

In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

In [4]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [5]:
train_transformer = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transformer = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [6]:
data_dir = "dataset/train"

In [7]:
apple_dataset = datasets.ImageFolder(root=data_dir, transform=train_transformer)

In [8]:
train_size = int(0.8 * len(apple_dataset))
val_size = len(apple_dataset) - train_size

In [9]:
train_data, val_data = random_split(apple_dataset, [train_size, val_size])

train_data.dataset.transform = train_transformer
val_data.dataset.transform = val_transformer

In [10]:
batch_size = 32

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

In [11]:
print(f"Найдено картинок: {len(apple_dataset)}")
print(f"Найдено классов: {len(apple_dataset.classes)}")
print(f"Имена классов: {apple_dataset.classes}")

images, labels = next(iter(train_loader))

print(f"Размерность картинок (X): {images.shape}") 
print(f"Размерность ответов (Y): {labels.shape}")

print(f"Для обучения (Train) выделено: {len(train_data)} картинок")
print(f"Для проверки (Validation) выделено: {len(val_data)} картинок")

Найдено картинок: 2537
Найдено классов: 4
Имена классов: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy']
Размерность картинок (X): torch.Size([32, 3, 224, 224])
Размерность ответов (Y): torch.Size([32])
Для обучения (Train) выделено: 2029 картинок
Для проверки (Validation) выделено: 508 картинок


In [12]:
class Classifier(nn.Module):
    def __init__(self, num_classes = 4):
        super(Classifier, self).__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )


        self.block3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.flatten = nn.Flatten()

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(64 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x
    
model = Classifier(num_classes=4)

In [13]:
raw_predictions = model(images)

print(f"Размерность ответа сети: {raw_predictions.shape}")
print(raw_predictions[0])

Размерность ответа сети: torch.Size([32, 4])
tensor([ 0.2776, -0.3825,  0.8413,  0.1601], grad_fn=<SelectBackward0>)


In [31]:
criterion= nn.CrossEntropyLoss().to(device)

lr = 0.001

optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

model= model.to(device)

In [ ]:
def train_and_validate(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=5):
    print(f"Начинаем обучение на {device}...")
    start_time = time.time()
    for epoch in range(num_epochs):
        print(f"\n--- Эпоха {epoch + 1}/{num_epochs} ---\n")
        model.train()

        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images = images.to(device).contiguous()
            labels = labels.to(device, dtype=torch.long).contiguous()

            optimizer.zero_grad()

            outputs = model(images).contiguous()

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)

            total_train += labels.size(0)

            correct_train += (predicted == labels).sum().item()
        
        train_acc = 100 * correct_train / total_train
        train_loss = running_loss / len(train_loader)
        model.eval()

        running_val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device).contiguous()
                labels = labels.to(device, dtype=torch.long).contiguous()

                outputs = model(images).contiguous()

                loss = criterion(outputs, labels)

                running_val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)

                total_val += labels.size(0)

                correct_val += (predicted == labels).sum().item()
        val_acc = 100 * correct_val / total_val
        val_loss = running_val_loss / len(val_loader)

        print(f"📚 Учеба (Train)  -> Loss: {train_loss:.3f} | Точность: {train_acc:.2f}%")
        print(f"🎯 Тест  (Val)    -> Loss: {val_loss:.3f} | Точность: {val_acc:.2f}%")

    print(f"\n🎉 Цикл завершен за {(time.time() - start_time):.1f} секунд!")
    return model

In [33]:
trained_model = train_and_validate(model, train_loader, val_loader, criterion, optimizer, device, num_epochs = 15)

Начинаем обучение на mps...

--- Эпоха 1/15 ---
📚 Учеба (Train)  -> Loss: 2.825 | Точность: 70.63%
🎯 Тест  (Val)    -> Loss: 0.418 | Точность: 85.04%

--- Эпоха 2/15 ---
📚 Учеба (Train)  -> Loss: 0.318 | Точность: 88.27%
🎯 Тест  (Val)    -> Loss: 0.235 | Точность: 91.54%

--- Эпоха 3/15 ---
📚 Учеба (Train)  -> Loss: 0.262 | Точность: 90.19%
🎯 Тест  (Val)    -> Loss: 0.279 | Точность: 89.96%

--- Эпоха 4/15 ---
📚 Учеба (Train)  -> Loss: 0.181 | Точность: 93.64%
🎯 Тест  (Val)    -> Loss: 0.298 | Точность: 90.35%

--- Эпоха 5/15 ---
📚 Учеба (Train)  -> Loss: 0.161 | Точность: 93.69%
🎯 Тест  (Val)    -> Loss: 0.561 | Точность: 85.24%

--- Эпоха 6/15 ---
📚 Учеба (Train)  -> Loss: 0.148 | Точность: 95.22%
🎯 Тест  (Val)    -> Loss: 0.163 | Точность: 93.31%

--- Эпоха 7/15 ---
📚 Учеба (Train)  -> Loss: 0.120 | Точность: 95.91%
🎯 Тест  (Val)    -> Loss: 0.155 | Точность: 93.50%

--- Эпоха 8/15 ---
📚 Учеба (Train)  -> Loss: 0.121 | Точность: 95.52%
🎯 Тест  (Val)    -> Loss: 0.156 | Точность: 94.

In [36]:
torch.save(model.state_dict(), "weights/apple_disease_model.pth")
print("Веса модели успешно сохранены!")

Веса модели успешно сохранены!
